# 05 · Cuantización de la ISI y la CD

**Objetivo.** Elegir los anchos de palabra del FIR de la ISI en punto fijo (paso C3.1 de la guía), con un criterio numérico:

- partir de la decisión 32: entrada `ac_fixed<14, 1>` (ADC), coeficientes `ac_fixed<18, 1>` (puerto de 18 bits del DSP48E2), productos a precisión completa, acumulador con log₂K bits de crecimiento y salida de 14 bits (DAC);
- barrer los bits de los **coeficientes** y del **acumulador**;
- separar las fuentes de error: **referencia** (FIR en flotante con la entrada sin cuantizar) → **flotante con la entrada cuantizada** → **punto fijo**;
- comprobar el criterio de la decisión 27 para la ISI: NRMSE ≤ 1e-3 contra la salida en flotante.

El modelo bit a bit es `emulador/fixed_point/isi_fir.py` (`isi_fir_fixed`), construido sobre `quantization.py` y aritmética entera, con la semántica de `ac_fixed` (tests: `tests/test_fixed_point_isi.py`; el cruce Python ↔ HALCON está en `tests/test_quant_crosscheck.py`, C3.0). La CD (C3.2) está en la parte II: primero con la FFT en flotante (C3.2.1) y después con la réplica bit a bit de la FFT (C3.2.2 y C3.2.3).

In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np

from emulador import plots
from emulador.blocks import ISI
from emulador.config import Scenario
from emulador.fixed_point import isi_fir_fixed
from emulador.isi import isi_apply
from emulador.metrics import nrmse
from emulador.quantization import quantize
from emulador.stimuli import comm_signal

plots.use_style()

## 1. Modelo en punto fijo

$$
y[n] = \sum_{k=0}^{K-1} h[k]\,x[n-k]
$$

| Punto | Formato de partida (decisión 32) | Modo (decisión 26) |
|---|---|---|
| Entrada x (ADC) | `<14, 1>`: paso 2⁻¹³, rango [−1, 1) | `AC_RND_CONV`, `AC_SAT` |
| Coeficientes h[k] | `<18, 1>`: paso 2⁻¹⁷ | `AC_RND_CONV`, `AC_SAT` (+1 no es representable: satura a 1 − 2⁻¹⁷) |
| Productos reales | `<32, 2>`, exactos | — |
| Acumulador | `<W_acc, I_acc>`, I_acc = 1 + 1 + 1 + ⌈log₂K⌉ (productos, suma compleja, crecimiento) | precisión completa (30 bits fraccionarios) o recortado a F bits por producto |
| Salida (DAC) | 14 bits, `<14, I_out>` | un solo redondeo `AC_RND_CONV`, `AC_SAT` |

La decisión 32 deja abierto I_out: la sección 2 muestra que con `<14, 1>` la salida satura.

**Fuentes de error** (cada NRMSE contra la referencia, salvo el interno):

- *entrada*: FIR flotante con la entrada cuantizada a 14 bits;
- *coeficientes*: FIR flotante con h cuantizado;
- *salida*: la referencia cuantizada al formato de salida;
- *interno* (lo que agrega el bloque): punto fijo contra el FIR flotante **con la misma entrada cuantizada**;
- *total*: punto fijo con la entrada sin cuantizar contra la referencia.

**Escenario.** 32 GBd, 2 sps (64 GSa/s), 16-QAM con RRC β = 0,1, 20 000 símbolos y 3 semillas (se promedia el NRMSE). Entrada escalada a **pico de componente 0,95**, la convención del PDF de arquitectura de la CD. Perfiles: el principal, Bessel m = 4 con B = 0,6·R_s y K = 16, y el de K = 32 de la regla de la decisión 6, Bessel m = 4 con B = 0,4·R_s.

In [ ]:
esc = Scenario()  # 32 GBd, 2 sps, roll-off 0,1
SEEDS = (1, 2, 3)
N_SYM = 20_000
PEAK = 0.95
X_FMT, H_FMT = (14, 1), (18, 1)
PROFILES = {
    "K = 16 (Bessel, B = 0,6·Rs)": ISI(profile="bessel", B_rel=0.6, order=4, K=16).h,
    "K = 32 (Bessel, B = 0,4·Rs)": ISI(profile="bessel", B_rel=0.4, order=4, K=32).h,
}
CRITERION = 1e-3  # decisión 27, ISI


def signal(seed, peak=PEAK):
    x = comm_signal(esc, "qam16", n_symbols=N_SYM, seed=seed).samples
    return x * peak / np.max(np.abs(np.r_[x.real, x.imag]))


SIGNALS = {s: signal(s) for s in SEEDS}


def mean_over_seeds(f):
    # f(x) -> float o tupla de floats; promedia sobre las semillas
    return np.mean([f(SIGNALS[s]) for s in SEEDS], axis=0)


for name, h in PROFILES.items():
    print(f"{name}: K = {h.size}, Σ|h| = {np.sum(np.abs(h)):.3f}, max|h| = {np.max(np.abs(h)):.3f}")
x = SIGNALS[1]
print(f"señal: {x.size} muestras, rms por componente {np.std(x.real):.3f}, pico/rms {20*np.log10(PEAK/np.std(x.real)):.1f} dB")

## 2. Punto de partida y formato de salida

In [ ]:
def breakdown(x, h, out_fmt, **kw):
    ref = isi_apply(x, h)
    xq = quantize(x, *X_FMT)
    fx_total = isi_fir_fixed(x, h, out_fmt=out_fmt, **kw)
    fx_int = isi_fir_fixed(xq, h, out_fmt=out_fmt, **kw)
    sat = fx_total.saturations
    return (
        nrmse(isi_apply(xq, h), ref),
        nrmse(isi_apply(x, quantize(h, *kw.get("h_fmt", H_FMT))), ref),
        nrmse(quantize(ref, *out_fmt), ref),
        nrmse(fx_int.y, isi_apply(xq, h)),
        nrmse(fx_total.y, ref),
        sat["out"],
        np.max(np.abs(np.r_[ref.real, ref.imag])),
    )


rows = []
print(f"{'perfil':30s} {'salida':8s} {'entrada':>9s} {'coef':>9s} {'salida':>9s} {'interno':>9s} {'total':>9s} {'sat. sal.':>9s} {'pico sal.':>9s}")
for name, h in PROFILES.items():
    for I_out in (1, 2):
        r = mean_over_seeds(lambda x: breakdown(x, h, (14, I_out)))
        rows.append((name, I_out, *r))
        print(f"{name:30s} <14,{I_out}>  " + " ".join(f"{v:9.2e}" for v in r[:5]) + f" {r[5]:9.1f} {r[6]:9.3f}")

**Lectura.**

- Con la entrada a pico 0,95, la salida del Bessel tiene **sobrepicos por encima de 1** (columna "pico sal."). Con salida `<14, 1>` esas muestras saturan (25 a 33 componentes por señal) y el error total queda entre 1,5 y 1,8·10⁻³: **no cumple** el 1e-3 de la decisión 27.
- Con salida `<14, 2>` no satura y el total baja a unos 2·10⁻⁴, dominado por la salida de 14 bits (un bit menos de resolución que `<14, 1>`) y la entrada.
- Los coeficientes de 18 bits y el acumulador de precisión completa aportan un error despreciable: el interno es prácticamente el de la salida.

La alternativa a `<14, 2>` es dejar más margen en la entrada. La sección 3 la mide.

## 3. Escala de pico de la entrada

Error total contra la referencia en función del pico de componente de la entrada, con salida `<14, 1>` y `<14, 2>`. Menos pico deja margen para los sobrepicos de la salida, pero cada componente usa menos del rango del ADC.

In [ ]:
PEAKS = np.array([0.5, 0.6, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0 - 2**-13])
h16 = PROFILES["K = 16 (Bessel, B = 0,6·Rs)"]
peak_sweep = {}
for I_out in (1, 2):
    vals = []
    for p in PEAKS:
        sig = {s: SIGNALS[s] * p / PEAK for s in SEEDS}
        e = np.mean([nrmse(isi_fir_fixed(sig[s], h16, out_fmt=(14, I_out)).y, isi_apply(sig[s], h16)) for s in SEEDS])
        n_sat = np.mean([isi_fir_fixed(sig[s], h16, out_fmt=(14, I_out)).saturations["out"] for s in SEEDS])
        vals.append((e, n_sat))
    peak_sweep[I_out] = np.array(vals)

fig, ax = plt.subplots(figsize=(6.5, 4))
for I_out, v in peak_sweep.items():
    ax.semilogy(PEAKS, v[:, 0], "o-", label=f"salida <14, {I_out}>")
ax.axhline(CRITERION, color="k", ls="--", lw=1, label="criterio 1e-3 (decisión 27)")
ax.set_xlabel("pico de componente de la entrada")
ax.set_ylabel("NRMSE total")
ax.set_title("ISI K = 16: error total contra el pico de entrada")
ax.legend()
plt.show()

print("pico   " + "  ".join(f"<14,{I}>: NRMSE  sat." for I in peak_sweep))
for i, p in enumerate(PEAKS):
    print(f"{p:.3f}  " + "  ".join(f"{peak_sweep[I][i, 0]:9.2e} {peak_sweep[I][i, 1]:5.1f}" for I in peak_sweep))

**Lectura.**

- Con `<14, 1>` el error es mínimo con pico de entrada 0,85 (1,4·10⁻⁴, mejor que `<14, 2>` con 0,95, porque la salida conserva el bit de resolución). Pero el margen es angosto: con 0,9 ya saturan unas pocas muestras y el error se triplica, y con 0,95 no cumple. El pico de salida depende del perfil (sobrepico del Bessel de 1,086 respecto del pico de entrada); un perfil con más sobrepico, como el Butterworth de estrés, necesitaría todavía menos pico.
- Con `<14, 2>` no satura en todo el barrido y el error baja poco a poco con el pico (de 3,8·10⁻⁴ a 1,9·10⁻⁴). Es la opción robusta: no depende del perfil cargado.
- Las dos cumplen el 1e-3 si la escala está bien elegida. La decisión de formato de salida y de escala de pico es del equipo (sección 6).

## 4. Bits de los coeficientes

Barrido de W_h con I_h = 1 (paso 2^−(W_h−1)), entrada `<14, 1>`, acumulador de precisión completa y salida `<14, 2>`. Se grafica:

- el NRMSE de los propios coeficientes, ‖h_q − h‖ / ‖h‖;
- el error interno con salida de 14 bits (incluye el piso de la salida);
- la contribución de los coeficientes sola: FIR flotante con h cuantizado contra h exacto, con la misma entrada.

In [ ]:
OUT_FMT = (14, 2)
W_HS = np.arange(8, 21)
coef_sweep = {}
for name, h in PROFILES.items():
    vals = []
    for Wh in W_HS:
        hq = quantize(h, Wh, 1)
        e_h = nrmse(hq, h)
        e_int = mean_over_seeds(lambda x: nrmse(isi_fir_fixed(quantize(x, *X_FMT), h, h_fmt=(Wh, 1), out_fmt=OUT_FMT).y, isi_apply(quantize(x, *X_FMT), h)))
        e_coef = mean_over_seeds(lambda x: nrmse(isi_apply(quantize(x, *X_FMT), hq), isi_apply(quantize(x, *X_FMT), h)))
        vals.append((e_h, e_int, e_coef))
    coef_sweep[name] = np.array(vals)

fig, axs = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, (name, v) in zip(axs, coef_sweep.items()):
    ax.semilogy(W_HS, v[:, 0], "s-", label="coeficientes ‖h_q − h‖/‖h‖")
    ax.semilogy(W_HS, v[:, 1], "o-", label="interno (salida <14, 2>)")
    ax.semilogy(W_HS, v[:, 2], "^-", label="solo coeficientes")
    ax.axhline(CRITERION, color="k", ls="--", lw=1, label="criterio 1e-3")
    ax.axvline(18, color="0.6", ls=":", lw=1)
    ax.set_xlabel("W_h [bits] (I_h = 1)")
    ax.set_title(name)
axs[0].set_ylabel("NRMSE")
axs[0].legend(fontsize=8)
plt.show()

for name, v in coef_sweep.items():
    floor = v[-1, 1]
    ok = W_HS[v[:, 1] <= CRITERION]
    near = W_HS[v[:, 1] <= 1.1 * floor]
    print(f"{name}: piso interno {floor:.2e}; W_h mínimo con interno ≤ 1e-3: {ok.min()}; con interno ≤ 1,1 × piso: {near.min()}")

## 5. Acumulador

**Crecimiento.** El acumulador reserva I_acc = 3 + ⌈log₂K⌉ bits enteros (incluido el signo). Se compara con lo que la señal usa de verdad: el máximo |parte real o imaginaria| del acumulador.

**Bits fraccionarios.** Con precisión completa el acumulador tiene F = 13 + 17 = 30 bits fraccionarios y es exacto. Si cada producto se recorta a F bits antes de sumar (redondeo a par o truncado), el acumulador es más angosto: W_acc = I_acc + F. El mínimo es F = 12, los bits fraccionarios de la salida `<14, 2>`.

In [ ]:
for name, h in PROFILES.items():
    res = [isi_fir_fixed(SIGNALS[s], h, out_fmt=OUT_FMT) for s in SEEDS]
    W_acc, I_acc = res[0].acc_fmt
    peak = max(r.acc_peak for r in res) * 2.0**-(W_acc - I_acc)
    used = int(np.ceil(np.log2(peak))) + 1 if peak > 0 else 1
    print(f"{name}: acumulador <{W_acc}, {I_acc}> (rango ±{2**(I_acc-1)}); pico usado {peak:.3f} -> {used} bits enteros con signo; margen {I_acc - used} bits")

ACC_FRACS = np.arange(12, 31)
acc_sweep = {}
for name, h in PROFILES.items():
    vals = []
    for F in ACC_FRACS:
        e = [
            mean_over_seeds(lambda x: nrmse(isi_fir_fixed(quantize(x, *X_FMT), h, acc_frac=int(F), acc_rounding=mode, out_fmt=OUT_FMT).y, isi_apply(quantize(x, *X_FMT), h)))
            for mode in ("half_even", "trunc")
        ]
        vals.append(e)
    acc_sweep[name] = np.array(vals)

fig, axs = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, (name, v) in zip(axs, acc_sweep.items()):
    I_acc = isi_fir_fixed(SIGNALS[1][:10], PROFILES[name], out_fmt=OUT_FMT).acc_fmt[1]
    ax.semilogy(ACC_FRACS + I_acc, v[:, 0], "o-", label="producto redondeado a par")
    ax.semilogy(ACC_FRACS + I_acc, v[:, 1], "s-", label="producto truncado")
    ax.axhline(CRITERION, color="k", ls="--", lw=1, label="criterio 1e-3")
    ax.set_xlabel(f"W_acc [bits] (I_acc = {I_acc})")
    ax.set_title(name)
axs[0].set_ylabel("NRMSE interno (salida <14, 2>)")
axs[0].legend(fontsize=8)
plt.show()

for name, v in acc_sweep.items():
    I_acc = isi_fir_fixed(SIGNALS[1][:10], PROFILES[name], out_fmt=OUT_FMT).acc_fmt[1]
    floor = v[-1, 0]
    for j, mode in enumerate(("redondeo a par", "truncado")):
        near = ACC_FRACS[v[:, j] <= 1.1 * floor]
        print(f"{name}, {mode}: F mínimo con interno ≤ 1,1 × piso ({floor:.2e}): {near.min()} -> W_acc = {near.min() + I_acc}")

## 6. Resumen y propuesta

Los números de las secciones 2 a 5 (promedio de 3 semillas, 20 000 símbolos, 2 sps):

In [ ]:
for name, h in PROFILES.items():
    r = mean_over_seeds(lambda x: breakdown(x, h, OUT_FMT))
    print(f"{name}, salida <14, 2>, h <18, 1>, acumulador completo:")
    print(f"   entrada 14 bits {r[0]:.2e} | coeficientes {r[1]:.2e} | salida {r[2]:.2e} | interno {r[3]:.2e} | total {r[4]:.2e}")
    print(f"   criterio de la decisión 27 (≤ 1e-3 contra el flotante): interno {'cumple' if r[3] <= CRITERION else 'NO cumple'}, total {'cumple' if r[4] <= CRITERION else 'NO cumple'}")

**Propuesta** (para la fila de formatos de la ISI en §1; la decide Wanda con el equipo):

- **Salida `<14, 2>`** con la entrada a pico 0,95 (como la CD): total 2,0·10⁻⁴ (K = 16) y 1,8·10⁻⁴ (K = 32), sin saturaciones con ningún pico. La alternativa es `<14, 1>` con la entrada a pico 0,85 (1,4·10⁻⁴), con poco margen y dependiente del perfil (sección 3). Con `<14, 1>` y pico 0,95 la salida satura y no cumple el criterio.
- **Coeficientes `<18, 1>`**: entran en el puerto de 18 bits del DSP48E2 y su error (7 a 9·10⁻⁶) queda muy por debajo del piso de la salida (sección 4). El criterio se cumple desde 11 a 12 bits y el piso se alcanza (≤ 1,1 × piso) desde 15 a 16 bits; con menos de 18 se ahorra poco, porque el multiplicador del DSP es el mismo.
- **Acumulador de precisión completa**: `<37, 7>` con K = 16 y `<38, 8>` con K = 32; entra en los 48 bits del acumulador del DSP48E2 sin recortar productos (sección 5). Recortado, alcanza el piso con F = 16 bits fraccionarios si se redondea cada producto (W_acc = 23 y 24) o con 18 a 19 si se trunca. El crecimiento reservado (3 + ⌈log₂K⌉ bits enteros) deja 5 y 6 bits de margen sobre lo que usa la señal: el peor caso Σ|h| (1,5 y 1,7) necesita solo 1 bit más que la entrada, pero achicarlo no ahorra DSP.
- La entrada `<14, 1>` es la del ADC y no se barre.

# Parte II · CD

## 7. CD con la FFT en flotante (C3.2.1)

Primera aproximación de la decisión 32 para la CD, **sin réplica de la FFT**: se cuantizan los puntos que no dependen de la arquitectura y el overlap-save corre en flotante.

| Punto | Formato | Modo |
|---|---|---|
| Entrada (ADC) | `<14, 1>`, pico por componente 0,95 (decisión 34) | `AC_RND_CONV`, `AC_SAT` |
| H̄[k] | `<18, 1>` | `AC_RND`, `AC_SAT` (tabla, decisión 32) |
| FFT, ×H̄, IFFT | flotante (`cd_overlap_save`) | — |
| Salida (DAC) | `<14, 2>` | `AC_RND_CONV`, `AC_SAT` |

**Errores** (NRMSE en el interior de la señal, descartando N muestras en cada borde; promedio de 3 semillas):

- *método*: overlap-save flotante contra `cd_reference`, con la entrada sin cuantizar;
- *ADC*: overlap-save con la entrada cuantizada contra la misma sin cuantizar. Es el error de cuantizar la entrada a 14 bits, la vara del error interno (decisión 27);
- *ADC + DAC*: además, la salida cuantizada a `<14, 2>`. Es el piso de los conversores (fila de §6);
- *H̄*: overlap-save con H̄ cuantizado contra H̄ exacto, con la entrada cuantizada. Es el error interno de esta aproximación (la FFT es exacta);
- *total*: cadena completa (entrada, H̄ y salida cuantizadas) contra `cd_reference` con la entrada sin cuantizar.

Criterios de la decisión 27: total < 1e-2 (objetivo 1e-3) e interno ≤ ½ del error del ADC.

**Escenarios.** 16-QAM con **RRC β = 0,1**, 20 000 símbolos, 3 semillas de la PRBS, D = 17 ps/(nm·km) (y un caso con 20), λ = 1550 nm, S = 0. La señal de 1,2 sps se arma como en los notebooks 07: a 6 sps y decimando por 5 (función local `transmit`, sin `Scenario`, que no llega a 480 GBd; B5.3 la llevará a `stimuli/`). La forma de la señal no depende de R_s; solo cambia f_s = 1,2·R_s.

In [ ]:
from emulador.blocks import ChromaticDispersion
from emulador.cd import cd_R, cd_overlap_save, cd_reference
from emulador.common import rrc_taps
from emulador.config import D_si
from emulador.stimuli import map_gray, prbs

UP, DOWN = 6, 5      # 6 sps -> /5 -> 1,2 sps
ROLLOFF, SPAN = 0.1, 64
N_SYM_CD = 20_000
SEEDS_CD = (1, 2, 3)
LAM_NM = 1550.0
H_FMT_CD, X_FMT_CD, OUT_FMT_CD = (18, 1), (14, 1), (14, 2)


def transmit(seed, n_sym=N_SYM_CD):
    # PRBS-15 -> Gray 16-QAM -> RRC a 6 sps -> 1,2 sps, pico por componente 0,95 (decisión 34)
    sym = map_gray(prbs(15, 4 * n_sym, seed=seed), "qam16")
    up = np.zeros(n_sym * UP, complex)
    up[::UP] = sym
    x = np.convolve(up, rrc_taps(ROLLOFF, SPAN, UP), mode="same")[::DOWN]
    return x * PEAK / np.max(np.abs(np.r_[x.real, x.imag]))


def os_aligned(x, Hbar):
    # overlap-save con la salida alineada (como ChromaticDispersion.process)
    N = Hbar.size // 2
    d = N // 2
    return cd_overlap_save(np.r_[x, np.zeros(d, complex)], Hbar)[d : d + x.size]


CD_SIGNALS = {s: transmit(s) for s in SEEDS_CD}
x = CD_SIGNALS[1]
print(f"{x.size} muestras a 1,2 sps; rms por componente {np.std(x.real):.3f}; pico/rms {20*np.log10(PEAK/np.std(x.real)):.1f} dB")

In [ ]:
SCENARIOS = (
    [dict(N_CD=2048, fs=fs, z=z, D=17.0) for fs in (72e9, 288e9, 576e9) for z in (0, 2, 10, 20)]
    + [dict(N_CD=2048, fs=576e9, z=20, D=20.0)]
    + [dict(N_CD=512, fs=fs, z=z, D=17.0) for fs in (72e9, 288e9) for z in (0, 2, 10, 20)]
    + [dict(N_CD=512, fs=576e9, z=2, D=17.0)]
)


def cd_block(sc):
    return ChromaticDispersion(fs=sc["fs"], D_ps_nm_km=sc["D"], wavelength_nm=LAM_NM, length_km=sc["z"], N=sc["N_CD"])


def cd_float_fft(sc, x):
    blk = cd_block(sc)
    H = blk.Hbar
    Hq = quantize(H, *H_FMT_CD, rounding="half_up")
    xq = quantize(x, *X_FMT_CD)
    ref = cd_reference(x, sc["fs"], D_si(sc["D"]), LAM_NM * 1e-9, sc["z"] * 1e3)
    y = os_aligned(x, H)
    yq = os_aligned(xq, H)
    yqH = os_aligned(xq, Hq)
    out = quantize(yqH, *OUT_FMT_CD)
    # mismo paso que <14, 2> con 16 bits enteros más: lo que difiere es lo que saturó
    n_sat = int(np.count_nonzero(quantize(yqH, 14 + 16, 2 + 16) != out))
    sl = slice(blk.N, x.size - blk.N)
    e = lambda a, b: nrmse(a[sl], b[sl])
    return dict(
        method=e(y, ref), adc=e(yq, y), adc_dac=e(quantize(yq, *OUT_FMT_CD), y), hbar=e(yqH, yq),
        total=e(out, ref), sat=n_sat, peak_out=np.max(np.abs(np.r_[y.real, y.imag])),
    )


def fmt_sc(sc):
    return f"N_CD={sc['N_CD']:4d} fs={sc['fs']/1e9:3.0f} GSa/s z={sc['z']:2d} km D={sc['D']:.0f}"


KEYS = ("method", "adc", "adc_dac", "hbar", "total")
c321 = []
print(f"{'escenario':40s} {'R':>4s} {'método':>9s} {'ADC':>9s} {'ADC+DAC':>9s} {'H̄':>9s} {'total':>9s} {'sat':>4s} {'pico':>5s}  criterios")
for sc in SCENARIOS:
    R = cd_R(sc["fs"], D_si(sc["D"]), LAM_NM * 1e-9, sc["z"] * 1e3)
    rs = [cd_float_fft(sc, CD_SIGNALS[s]) for s in SEEDS_CD]
    m = {k: np.mean([r[k] for r in rs]) for k in KEYS}
    m["sat"] = np.mean([r["sat"] for r in rs])
    m["peak_out"] = max(r["peak_out"] for r in rs)
    ok = ("<1e-2 " if m["total"] < 1e-2 else "NO<1e-2 ") + ("<1e-3 " if m["total"] < 1e-3 else "NO<1e-3 ") + ("int≤½ADC" if m["hbar"] <= m["adc"] / 2 else "int>½ADC")
    c321.append((sc, R, m))
    print(f"{fmt_sc(sc):40s} {R:4d} " + " ".join(f"{m[k]:9.2e}" for k in KEYS) + f" {m['sat']:4.0f} {m['peak_out']:5.2f}  {ok}")

In [ ]:
# R > N_CD: el bloque lo rechaza (cd_N, decisión 30)
bad = dict(N_CD=512, fs=576e9, z=20, D=17.0)
print(f"{fmt_sc(bad)}: R = {cd_R(bad['fs'], D_si(bad['D']), LAM_NM * 1e-9, bad['z'] * 1e3)}")
try:
    cd_block(bad)
except ValueError as err:
    print("ValueError:", err)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)
for ax, N_CD in zip(axs, (2048, 512)):
    rows_ = [(sc, R, m) for sc, R, m in c321 if sc["N_CD"] == N_CD]
    labels = [f"{sc['fs']/1e9:.0f}/{sc['z']}" + ("*" if sc["D"] != 17 else "") for sc, _, _ in rows_]
    xs = np.arange(len(rows_))
    for k, lab, mk in (("method", "método", "x"), ("adc", "ADC 14 bits", "o"), ("adc_dac", "ADC + DAC", "s"), ("hbar", "interno (H̄ 18 bits)", "^"), ("total", "total", "D")):
        ax.semilogy(xs, [m[k] for _, _, m in rows_], mk + "-", label=lab, lw=1)
    ax.axhline(1e-2, color="k", ls="--", lw=1)
    ax.axhline(1e-3, color="k", ls=":", lw=1)
    ax.set_xticks(xs, labels, rotation=45, fontsize=8)
    ax.set_xlabel("f_s [GSa/s] / z [km]  (* D = 20)")
    ax.set_title(f"N_CD = {N_CD}: FFT en flotante")
axs[0].set_ylabel("NRMSE")
axs[0].legend(fontsize=8)
plt.show()

**Lectura (C3.2.1).**

- **Pisos de los conversores** (decisión 27 y fila de §6): cuantizar la entrada a `<14, 1>` con pico 0,95 da **1,05·10⁻⁴** en todos los escenarios (el PDF de arquitectura daba 1,1·10⁻⁴), y la entrada más la salida `<14, 2>` dan **2,34·10⁻⁴** (el PDF daba 2,4·10⁻⁴, citado del notebook 08 borrado). **Caso límite del piso ADC + DAC: z = 0**, donde sube a 2,78·10⁻⁴. Es esperable: la salida es la entrada misma, sobre la grilla de 2⁻¹³, y al pasarla a la grilla de 2⁻¹² la mitad de las muestras cae justo en un empate. El error deja de ser uniforme y vale ±q/2 o 0.
- **H̄ de 18 bits** agrega unos 3·10⁻⁶ (5·10⁻⁶ con z = 0): unas 30 veces menos que la mitad del error del ADC (5·10⁻⁵). Cumple el criterio del error interno con mucho margen; el presupuesto que queda es para la FFT (C3.2.2 y C3.2.3).
- **Salida `<14, 2>`**: el pico por componente de la salida llega a 1,72 (576 GSa/s, 2 km): la CD reparte la energía y crea picos más altos que los de la entrada (0,95). No satura en ningún escenario; el margen hasta 2 es de 1,3 dB.
- **Total** contra `cd_reference`: entre 2,3·10⁻⁴ y 6,7·10⁻⁴, **por debajo del objetivo de 1e-3** en todos los escenarios. Con N_CD = 2048 lo dominan los conversores; con N_CD = 512 y R cerca del máximo (288 GSa/s y 20 km, R = 225) ya pesa el error del método (6,2·10⁻⁴).
- R > N_CD (576 GSa/s, 20 km, N_CD = 512: R = 903) lo rechaza el bloque, como pide la decisión 30.

## 8. Réplica bit a bit de la FFT (C3.2.3)

La cadena completa en punto fijo de `emulador/fixed_point/cd_fft.py` (`cd_overlap_save_fixed`): ADC `<14, 1>` → datos `<W_d, 2>` → FFT DIF radix-2² con escalado "mitad + 1" → ×H̄ `<18, 1>` en bit invertido → IFFT DIT → DAC `<14, 2>`, con los modos de la decisión 32 (mariposas `AC_RND_CONV`, productos `AC_RND`, `AC_SAT` en todo). La réplica está verificada bit a bit contra una referencia con enteros de Python y, con formatos amplios, contra `cd_overlap_save` y `process_hw` (`tests/test_fixed_point_cd.py`).

**Errores** (mismo interior y mismas 3 semillas que en la sección 7):

- *interno*: salida de la IFFT (antes del DAC) contra `cd_overlap_save` en flotante **con la misma entrada cuantizada** y H̄ exacto. Incluye H̄, twiddles y redondeos de las etapas;
- *total*: salida del DAC contra `cd_reference` con la entrada sin cuantizar.

**Regla de decisión** (fijada por Wanda antes de correr): si el **peor** error interno con W_d = 20 (máximo sobre las 3 semillas y todos los escenarios, N_CD = 512 y 2048) es ≤ 5,25·10⁻⁵ (½ del piso del ADC, 1,05·10⁻⁴) y no hay saturaciones, **W_d = 20 queda confirmado**. Si no, se muestra en qué etapa se acumula el error y los resultados con W_d = 21 y 22.

In [ ]:
from emulador.fixed_point import CdFixedConfig, cd_overlap_save_fixed, stage_errors

W_DS = (16, 18, 20, 21, 22)
HALF_ADC = 1.05e-4 / 2  # regla de decisión: ½ del piso del ADC de la sección 7


def cd_fixed(sc, x, W_d):
    blk = cd_block(sc)
    xq = quantize(x, *X_FMT_CD)
    res = cd_overlap_save_fixed(x, blk.Hbar, CdFixedConfig(W_d=W_d))
    ref = cd_reference(x, sc["fs"], D_si(sc["D"]), LAM_NM * 1e-9, sc["z"] * 1e3)
    d = blk.delay_samples
    y_int = res.y_internal[d : d + x.size]  # alineada, como os_aligned
    y_out = res.y[d : d + x.size]
    sl = slice(blk.N, x.size - blk.N)
    return dict(
        internal=nrmse(y_int[sl], os_aligned(xq, blk.Hbar)[sl]),
        total=nrmse(y_out[sl], ref[sl]),
        sat=res.saturations,
    )


c323 = {}
for i_sc, sc in enumerate(SCENARIOS):
    for W_d in W_DS:
        c323[i_sc, W_d] = [cd_fixed(sc, CD_SIGNALS[s], W_d) for s in SEEDS_CD]

worst20 = max(r["internal"] for i in range(len(SCENARIOS)) for r in c323[i, 20])
sat20 = sum(sum(r["sat"].values()) for i in range(len(SCENARIOS)) for r in c323[i, 20])
print(f"W_d = 20: peor error interno = {worst20:.3e} (límite {HALF_ADC:.3e}); saturaciones = {sat20}")
print("=> W_d = 20 CONFIRMADO" if (worst20 <= HALF_ADC and sat20 == 0) else "=> NO cumple la regla: ver etapas y W_d = 21, 22")

In [ ]:
print(f"{'escenario':40s} {'R':>4s} | {'interno máx, W_d =':^29s} | {'total medio, W_d =':^29s} | sat 20")
print(f"{'':40s} {'':4s} | " + " ".join(f"{w:>9d}" for w in (20, 21, 22)) + " | " + " ".join(f"{w:>9d}" for w in (20, 21, 22)) + " |")
for i_sc, (sc, R, _) in enumerate(c321):
    imax = [max(r["internal"] for r in c323[i_sc, w]) for w in (20, 21, 22)]
    tmean = [np.mean([r["total"] for r in c323[i_sc, w]]) for w in (20, 21, 22)]
    s20 = sum(sum(r["sat"].values()) for r in c323[i_sc, 20])
    flag = "" if imax[0] <= HALF_ADC else "  > ½ ADC"
    print(f"{fmt_sc(sc):40s} {R:4d} | " + " ".join(f"{v:9.2e}" for v in imax) + " | " + " ".join(f"{v:9.2e}" for v in tmean) + f" | {s20:5d}{flag}")

**Barrido de W_d con L = 4096** (N_CD = 2048, todos sus escenarios y semillas): error interno medio y máximo contra W_d, y saturaciones por transformada.

In [ ]:
idx2048 = [i for i, sc in enumerate(SCENARIOS) if sc["N_CD"] == 2048]
fig, ax = plt.subplots(figsize=(6.5, 4))
for agg, lab, mk in ((np.max, "máximo", "o-"), (np.mean, "medio", "s--")):
    ax.semilogy(W_DS, [agg([r["internal"] for i in idx2048 for r in c323[i, w]]) for w in W_DS], mk, label=f"interno {lab}")
ax.semilogy(W_DS, [np.mean([r["total"] for i in idx2048 for r in c323[i, w]]) for w in W_DS], "^-", label="total medio")
ax.axhline(HALF_ADC, color="k", ls="--", lw=1, label="½ piso ADC (5,25·10⁻⁵)")
ax.axhline(1e-3, color="k", ls=":", lw=1, label="objetivo total 1e-3")
ax.set_xlabel("W_d [bits]")
ax.set_ylabel("NRMSE")
ax.set_title("CD bit a bit, L = 4096")
ax.legend(fontsize=8)
plt.show()

print("W_d  sat. in  fft  hbar  ifft  out   (suma sobre escenarios de N_CD = 2048 y semillas)")
for w in W_DS:
    tot = {k: sum(r["sat"][k] for i in idx2048 for r in c323[i, w]) for k in ("in", "fft", "hbar", "ifft", "out")}
    print(f"{w:3d}  " + " ".join(f"{tot[k]:5d}" for k in ("in", "fft", "hbar", "ifft", "out")))

**Dónde se acumula el error** (W_d = 20, el escenario con el peor error interno): NRMSE acumulado después de cada etapa contra el mismo grafo en flotante (`stage_errors`).

In [ ]:
i_w, s_w = max(((i, s) for i in range(len(SCENARIOS)) for s in range(len(SEEDS_CD))), key=lambda t: c323[t[0], 20][t[1]]["internal"])
sc_w = SCENARIOS[i_w]
se = stage_errors(CD_SIGNALS[SEEDS_CD[s_w]], cd_block(sc_w).Hbar, CdFixedConfig(W_d=20))
fig, ax = plt.subplots(figsize=(9, 3.8))
ax.semilogy(range(len(se)), [e for _, e, _ in se], "o-")
ax.axhline(HALF_ADC, color="k", ls="--", lw=1, label="½ piso ADC")
ax.set_xticks(range(len(se)), [n for n, _, _ in se], rotation=60, fontsize=8)
ax.set_ylabel("NRMSE acumulado")
ax.set_title(f"Error por etapa, W_d = 20: {fmt_sc(sc_w)}, semilla {SEEDS_CD[s_w]}")
ax.legend(fontsize=8)
plt.show()
print("saturaciones por etapa:", sum(n for _, _, n in se))

## 9. Pisos de los conversores y resumen de la CD

In [ ]:
adc = [m["adc"] for _, _, m in c321]
adc_dac = [m["adc_dac"] for sc, _, m in c321 if sc["z"] > 0]
adc_dac0 = [m["adc_dac"] for sc, _, m in c321 if sc["z"] == 0]
print(f"piso ADC (entrada <14, 1>, pico 0,95):        {min(adc):.2e} a {max(adc):.2e}")
print(f"piso ADC + DAC (salida <14, 2>), z > 0:       {min(adc_dac):.2e} a {max(adc_dac):.2e}")
print(f"piso ADC + DAC, caso límite z = 0:            {min(adc_dac0):.2e} a {max(adc_dac0):.2e}")
t20 = [np.mean([r["total"] for r in c323[i, 20]]) for i in range(len(SCENARIOS))]
print(f"total con W_d = 20: {min(t20):.2e} a {max(t20):.2e} (aceptación < 1e-2, objetivo < 1e-3)")

**Lectura (C3.2.3).**

- **W_d = 20 queda confirmado** por la regla de decisión: el peor error interno, sobre las 3 semillas y los 22 escenarios (N_CD = 512 y 2048), es **4,12·10⁻⁵** (≤ 5,25·10⁻⁵), y no hay ninguna saturación en ningún punto (ADC, FFT, ×H̄, IFFT, DAC). Coincide con la estimación previa (2,5 a 4,2·10⁻⁵). El margen es de 1,1 dB.
- El error interno casi no depende del escenario: entre 3,5 y 4,1·10⁻⁵ con L = 4096 y entre 3,0 y 3,7·10⁻⁵ con L = 1024 (menos etapas). Con z = 0 es un poco menor.
- **Dónde se acumula**: crece de a poco en cada etapa de la FFT y de la IFFT, sin una etapa que domine. La primera etapa no agrega error, porque la extensión de signo deja 5 bits en cero y el desplazamiento de 1 bit es exacto. Las etapas sin desplazamiento ni rotación no redondean, por eso hay escalones planos.
- **W_d más grande** baja el interno a la mitad por bit (2,2·10⁻⁵ con 21, 1,3·10⁻⁵ con 22), pero el **total casi no cambia** (2,4·10⁻⁴ → 2,35·10⁻⁴ en los escenarios típicos), porque lo dominan los conversores: la salida vuelve a 14 bits. Con W_d = 18 el interno (1,6·10⁻⁴) ya pasa la mitad del ADC, y con 16 (6·10⁻⁴) el total se acerca al objetivo de 1e-3.
- **Pisos de los conversores**, medidos por separado: ADC solo **1,05·10⁻⁴**; ADC + DAC **2,34·10⁻⁴** (z > 0) y 2,78·10⁻⁴ en el caso límite z = 0.
- **Total** con W_d = 20 contra `cd_reference`: 2,4 a 6,7·10⁻⁴, **por debajo del objetivo de 1e-3** en todos los escenarios. El peor es N_CD = 512 con R cerca del máximo (288 GSa/s, 20 km, R = 225), por el error del método.